# Longformer + Metadata Fusion -- IMDb Rating Predictor

**Architecture:** Script text -> Longformer CLS (768-dim) + Metadata FC (3->32) -> Fusion (800->128->1) -> Rating

**Before running:**
1. Go to **Runtime -> Change runtime type -> Select GPU** (T4)
2. Upload `scripts.zip` (your ~5204 .txt scripts zipped) to Google Drive root (`MyDrive/`)
3. Upload `movie_lengths.xlsx` to Google Drive root (`MyDrive/`)

The notebook will mount Drive, copy data, and run the full training pipeline.

In [ ]:
#@title 1. GPU Check
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print("  DEVICE INFORMATION")
print("=" * 60)
print(f"  Using device : {device}")
if torch.cuda.is_available():
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  VRAM         : {gpu_mem:.1f} GB")
else:
    print("  WARNING: No GPU detected!")
    print("  Go to Runtime -> Change runtime type -> Select GPU")
print("=" * 60)

In [ ]:
#@title 2. Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers tqdm openpyxl

In [ ]:
#@title 3. Clone Repo & Setup Data
import os

# -- Clone the repository --
REPO_URL = "https://github.com/YOUR_USERNAME/imdb-predictor.git"  # <-- EDIT THIS
BRANCH = "longformer-DL-approach"
WORK_DIR = "/content/imdb-predictor"

if not os.path.exists(WORK_DIR):
    !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
else:
    print(f"Repo already cloned at {WORK_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
#@title 4. Copy Data Files from Drive
import shutil
import zipfile

DRIVE_ROOT = "/content/drive/MyDrive"

# -- Copy Excel metadata --
excel_src = os.path.join(DRIVE_ROOT, "movie_lengths.xlsx")
if os.path.exists(excel_src):
    shutil.copy2(excel_src, "movie_lengths.xlsx")
    print("[OK] movie_lengths.xlsx copied")
else:
    print("[ERROR] movie_lengths.xlsx not found in Drive root!")
    print("  Please upload movie_lengths.xlsx to MyDrive/")

# -- Unzip scripts --
scripts_zip = os.path.join(DRIVE_ROOT, "scripts.zip")
if os.path.exists(scripts_zip) and not os.path.exists("scripts"):
    print("Extracting scripts.zip ...")
    with zipfile.ZipFile(scripts_zip, 'r') as z:
        z.extractall(".")
    print(f"[OK] Extracted. scripts/ has {len(os.listdir('scripts'))} files")
elif os.path.exists("scripts"):
    print(f"[OK] scripts/ already exists ({len(os.listdir('scripts'))} files)")
else:
    print("[ERROR] scripts.zip not found in Drive root!")
    print("  Please zip your scripts/ folder and upload scripts.zip to MyDrive/")

In [ ]:
#@title 5. Run Longformer Training
!python longformer_trainer.py